Coffee Shop Data set Cleaning 

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np


In [ ]:
# Define your base source directory path cleanly in the brackets
base_path = r""

In [ ]:
# Define the new target subfolder path
clean_folder_path = os.path.join(base_path, "clean dataset")

print("Step 1: Commencing environment cleanup...")

In [ ]:
# Wiping out the master excel file if it exists to keep your directory spotless
master_excel_path = os.path.join(base_path, "Coffee_Shop_Master_Cleaned.xlsx")
if os.path.exists(master_excel_path):
    os.remove(master_excel_path)
    print("-> Successfully deleted 'Coffee_Shop_Master_Cleaned.xlsx'.")

In [ ]:
# Creating the fresh 'clean dataset' folder safely
if os.path.exists(clean_folder_path):
    # Wiping old files inside the clean folder if running the script multiple times
    shutil.rmtree(clean_folder_path)
os.makedirs(clean_folder_path)
print(f"-> Fresh target folder created at: {clean_folder_path}\n")


print("Step 2: Processing, formatting, and saving standardized data files...")

In [ ]:
# --- 1. CLEANING: Sales Receipts ---
receipts = pd.read_csv(os.path.join(base_path, "201904 sales reciepts.csv"))
receipts.columns = receipts.columns.str.strip().str.replace('"', '')
# Combine Date and Time into a standard uniform DATETIME object
receipts['transaction_dt'] = pd.to_datetime(receipts['transaction_date'] + ' ' + receipts['transaction_time'])
# Sync date format exactly to standard ISO string format
receipts['transaction_date'] = pd.to_datetime(receipts['transaction_date']).dt.strftime('%Y-%m-%d')
receipts['customer_id'] = receipts['customer_id'].fillna(0).astype(int)
receipts['quantity'] = pd.to_numeric(receipts['quantity'], errors='coerce').fillna(0).astype(int)
receipts['unit_price'] = pd.to_numeric(receipts['unit_price'], errors='coerce').fillna(0.0)
receipts['line_item_amount'] = receipts['quantity'] * receipts['unit_price']
receipts.to_csv(os.path.join(clean_folder_path, 'cleaned_sales_receipts.csv'), index=False)
print("-> Processed and saved: cleaned_sales_receipts.csv")

In [ ]:
# --- 2. CLEANING: Product Master ---
product = pd.read_csv(os.path.join(base_path, "product.csv"))
product.columns = product.columns.str.strip()
product['current_retail_price'] = product['current_retail_price'].astype(str).str.replace('$', '', regex=False).str.strip()
product['current_retail_price'] = pd.to_numeric(product['current_retail_price'], errors='coerce')
product['current_wholesale_price'] = pd.to_numeric(product['current_wholesale_price'].astype(str).str.replace('$', '', regex=False).str.strip(), errors='coerce')
for col in ['tax_exempt_yn', 'promo_yn', 'new_product_yn']:
    product[col] = product[col].astype(str).str.upper().str.strip().map({'Y': 1, 'N': 0}).fillna(0).astype(int)
product.to_csv(os.path.join(clean_folder_path, 'cleaned_product.csv'), index=False)
print("-> Processed and saved: cleaned_product.csv")

In [ ]:
# --- 3. CLEANING: Customer Profiles ---
customer = pd.read_csv(os.path.join(base_path, "customer.csv"))
customer.columns = customer.columns.str.strip().str.replace('"', '')
customer['gender'] = customer['gender'].fillna('Unknown').str.strip().replace({'N': 'Unknown', '': 'Unknown'})
customer['birth_year'] = pd.to_numeric(customer['birth_year'], errors='coerce').fillna(0).astype(int)
customer.to_csv(os.path.join(clean_folder_path, 'cleaned_customer.csv'), index=False)
print("-> Processed and saved: cleaned_customer.csv")

In [ ]:
# --- 4. CLEANING: Pastry Inventory (Date Formatted Seamlessly) ---
inventory = pd.read_csv(os.path.join(base_path, "pastry inventory.csv"))
inventory.columns = inventory.columns.str.strip()
inventory['transaction_date'] = pd.to_datetime(inventory['transaction_date']).dt.strftime('%Y-%m-%d')
if '% waste' in inventory.columns:
    inventory['waste_pct'] = inventory['% waste'].astype(str).str.replace('%', '', regex=False).str.strip()
    inventory['waste_pct'] = pd.to_numeric(inventory['waste_pct'], errors='coerce').fillna(0) / 100.0
    inventory.drop(columns=['% waste'], inplace=True)
inventory.to_csv(os.path.join(clean_folder_path, 'cleaned_pastry_inventory.csv'), index=False)
print("-> Processed and saved: cleaned_pastry_inventory.csv")

In [ ]:
# --- 5. CLEANING: Dates Dimension (Date Formatted Seamlessly) ---
dates_df = pd.read_csv(os.path.join(base_path, "Dates.csv"))
dates_df.columns = dates_df.columns.str.strip()
dates_df['transaction_date'] = pd.to_datetime(dates_df['transaction_date']).dt.strftime('%Y-%m-%d')
dates_df.to_csv(os.path.join(clean_folder_path, 'cleaned_dates.csv'), index=False)
print("-> Processed and saved: cleaned_dates.csv")


In [ ]:
# --- 6. CLEANING: Generations Lookup ---
generations = pd.read_csv(os.path.join(base_path, "generations.csv"))
generations.columns = generations.columns.str.strip()
generations.to_csv(os.path.join(clean_folder_path, 'cleaned_generations.csv'), index=False)
print("-> Processed and saved: cleaned_generations.csv")

In [ ]:
# --- 7. CLEANING: Outlets, Staff, and Targets ---
for file_name, out_name in [('sales_outlet.csv', 'cleaned_sales_outlet.csv'), 
                            ('staff.csv', 'cleaned_staff.csv'), 
                            ('sales targets.csv', 'cleaned_sales_targets.csv')]:
    df = pd.read_csv(os.path.join(base_path, file_name))
    df.columns = df.columns.str.strip()
    string_cols = df.select_dtypes(include=['object']).columns
    for col in string_cols:
        df[col] = df[col].astype(str).str.strip()
    df.to_csv(os.path.join(clean_folder_path, out_name), index=False)
    print(f"-> Processed and saved: {out_name}")

print("\nPipeline execution complete! Check your new folder structure in your file explorer.")